# Spatial-Iteration-Engine – Control desde Jupyter

Cámara → filtros (Python o C++) → **ventana de preview** (sin VLC).

**Requisitos:**
- Ejecutar Jupyter desde la **raíz del repo** o desde `python/ascii_stream_engine/examples/`.
- Entorno con `pip install -r python/requirements.txt` y, si usas filtros C++, compilar: `./cpp/build.sh`.

**Pasos:** Ejecutar celdas en orden. Usar **Start** para ver la ventana; **Stop** para detener.

In [ ]:
import os
import sys

# Paths: repo root y cpp/build para módulos C++
cwd = os.getcwd()
if os.path.basename(cwd) == "examples":
    # Estamos en python/ascii_stream_engine/examples → repo = ../../..
    repo_root = os.path.abspath(os.path.join(cwd, "..", "..", ".."))
else:
    repo_root = os.path.abspath(cwd)

for path in [repo_root, os.path.join(repo_root, "python"), os.path.join(repo_root, "cpp", "build")]:
    if path not in sys.path and os.path.exists(path):
        sys.path.insert(0, path)

print("Repo:", repo_root)
print("cpp/build existe:", os.path.exists(os.path.join(repo_root, "cpp", "build")))

In [ ]:
from ascii_stream_engine import (
    EngineConfig,
    StreamEngine,
    OpenCVCameraSource,
    PassthroughRenderer,
    PreviewSink,
    FilterPipeline,
)
from ascii_stream_engine.adapters.processors import (
    BrightnessFilter,
    InvertFilter,
    CppBrightnessContrastFilter,
    CppInvertFilter,
    CppGrayscaleFilter,
    CppChannelSwapFilter,
)

config = EngineConfig(
    host="127.0.0.1",
    port=1234,
    fps=20,
    frame_buffer_size=0,
    sleep_on_empty=0.001,
)

camera_index = 0
filters = FilterPipeline([])

engine = StreamEngine(
    source=OpenCVCameraSource(camera_index),
    renderer=PassthroughRenderer(),
    sink=PreviewSink(),
    config=config,
    filters=filters,
)

print("Engine creado: cámara → PassthroughRenderer → PreviewSink")

In [ ]:
import ipywidgets as widgets
from IPython.display import display

status = widgets.HTML(value="<i>Parado. Pulsa Start para abrir la ventana de preview.</i>")

camera_input = widgets.IntText(value=0, description="Cámara", min=0, max=9)

FILTERS_OPTIONS = {
    "(ninguno)": [],
    "Brillo/Contraste (C++)": [CppBrightnessContrastFilter(brightness_delta=10, contrast_factor=1.1)],
    "Invert (C++)": [CppInvertFilter()],
    "Gris (C++)": [CppGrayscaleFilter()],
    "Brillo + Invert (C++)": [
        CppBrightnessContrastFilter(brightness_delta=10, contrast_factor=1.1),
        CppInvertFilter(),
    ],
    "Brillo (Python)": [BrightnessFilter()],
    "Invert (Python)": [InvertFilter()],
}

filter_dd = widgets.Dropdown(
    options=list(FILTERS_OPTIONS.keys()),
    value="(ninguno)",
    description="Filtros",
)

start_btn = widgets.Button(description="Start", button_style="success")
stop_btn = widgets.Button(description="Stop", button_style="danger")

def on_start(_):
    idx = int(camera_input.value)
    source = engine.get_source()
    if hasattr(source, "set_camera_index"):
        source.set_camera_index(idx)
    flist = FILTERS_OPTIONS[filter_dd.value]
    engine.filter_pipeline.replace(flist)
    engine.start()
    status.value = f"<b>Corriendo</b> – Cámara {idx}, filtros: {filter_dd.value}. Ventana abierta."

def on_stop(_):
    engine.stop()
    status.value = "<i>Parado.</i>"

start_btn.on_click(on_start)
stop_btn.on_click(on_stop)

panel = widgets.VBox([
    widgets.HTML("<b>Preview (ventana OpenCV)</b>"),
    widgets.HBox([start_btn, stop_btn]),
    camera_input,
    filter_dd,
    status,
])
display(panel)

**Notas:**
- La ventana de preview puede aparecer detrás del navegador; búscala en la barra de tareas.
- Para cambiar cámara o filtros: pulsa **Stop**, cambia valores y vuelve a **Start**.
- Si la cámara no enciende, prueba índice **2** (común en Linux).